# <center>CMPBIO293
# <center> Numerical Methods of Solving ODEs
## <center>Denis Titov

**The goal of this jupyter notebook is to introduce you to how numerical methods for solution of ordinary differential equations (ODEs) work in general and review various option of SciPy solve_ivp() function. This is a math heavy subject but it's good to have a basic idea of how ODE solvers like solve_ivp() actually work if you plan to use them.**

In [ ]:
# Symbolic math
import sympy as sym
from sympy import Eq

# Array manipulation
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# Numerical solution for differential equations
from scipy.integrate import solve_ivp

# Example: Clearance of drugs from the blood

We will use drug clearance from the blood through kidney filtration as an example in this section. Assume that a drug is cleared from the blood at a rate proportional to the concentration of drug in the blood

Therefore, drug clearance be describe using differential equation $\frac{d[Drug](t)}{dt}=-k[Drug](t)$, where $t$ is time,  k is some constant describing kindey filtration rate,  $[Drug](t)$ is blood concentration of drug at time $t$ and $[Drug]_0$ is is blood concentration of drug at $t=0$. 

In [ ]:
# Differential equation as a Python function?
# args:
#   t: Time
#   d: Drug concentration at time t
# returns:
#   change in drug concentration
def clearance_rate(t, d, k=1):
    # TODO: Evaluate differential equation for a given drug concentration and time

# Euler method

The basic idea behind numerical methods for solving ODEs is to exchange $dt$ for some small number for $\Delta t$ and do repetitive algebraic calculation with it.  

For example, let't take some value of $\Delta t$ and find a solution for $\frac{d[Drug](t)}{dt}=-k[Drug](t)$ starting from initial concentration $[Drug]_0$  
  
First, we rewrite $\frac{d[Drug](t)}{dt}=-k[Drug](t)$ as $\frac{\Delta[Drug]}{\Delta t}=\frac{[Drug]_{t+\Delta t}-[Drug]_{t}}{\Delta t}=-k[Drug]_{t}$.  
  
Next, we can use our initial condition $[Drug]_{t=0}=[Drug]_0$ to find $[Drug]_{\Delta t}$:  
  
$[Drug]_{\Delta t}=[Drug]_0-\Delta t \cdot k \cdot [Drug]_0=[Drug]_0 \cdot (1-\Delta t \cdot k)$  
  
Repeat the same to get $[Drug]_{2 \cdot \Delta t}$, $[Drug]_{3 \cdot \Delta t}$ ... etc:  
  
$[Drug]_{2 \cdot \Delta t}=[Drug]_{\Delta t}-\Delta t \cdot k \cdot [Drug]_{\Delta t}=[Drug]_0 \cdot (1-\Delta t \cdot k)^2$  
  
$[Drug]_{3 \cdot \Delta t}=[Drug]_0 \cdot (1-\Delta t \cdot k)^3$... etc etc etc for as many $\Delta t$ steps as we want.   
  
This is called the explicit Euler method and is the simplest method that can be used to numerically solve ODEs although it is rarely used in practice as there are more efficient methods.  

### Let's implement Euler method with python code.

#### For loops

We will need to use a *for loop* to implement Euler method.  
*For loops* are very common in computer programs and is a way to make computer do a repetitive task like Euler method.  
Here's a simple example of a *for loop*:

In [ ]:
for i in range(5):
    print(i)

Note that numbering in python starts at 0 as opposed to 1 and that last number (i.e. 5 above) is not included.  
*Side note: this is a major point of contention betweem programming languages with more math/science focused languages having 1-based indexing (e.g, Fortran, Julia, Matlab, R) while other languages made a poor choice of having 0 based indexing which is in my humble opinion nonsensical! (e.g., C, Python, Java)*

#### Euler method code

In [ ]:
# args:
#   rate: differential equation for drug concentration (function)
#   t: current time
#   d: current drug concetration and step size
# returns: 
#   next time, next concentration
def euler_step(rate, t, d, dt):
    # TODO: Single Euler step

In [ ]:
# Delta t or step size
dt = 0.5

# Initalize array containing drug concentration and time
Drug = np.array([1,])
time = np.array([0,])
    
# TODO: Calculate euler method for each time step

# Plot drug concentration vs time
plt.plot(time, Drug, label='Euler')
plt.legend();

What happens if $\Delta t$=2?


What happens if $\Delta t$>2?

### What is the advantage of inventing other numerical method if Euler works?
There're hundreds of numerical methods that people have invented.  
Numerical methods have to balance numerical accuracy vs computational speed.  
Euler method will be 10x more accurate if you decrease $\Delta t$ 10x but it will also take 10x longer for computer to calculate a solution because it'll have to take 10x more steps. Because of this tradeoff between speed and accuracy, people have spent lots of time inventing new methods that can be as accurate as Euler while taking much larger steps or much faster than Euler with the same accuracy.  More modern methods are orders of magnitude better than Euler as we'll se below.

# Numerical solution of $\frac{d[Drug]}{dt}=-k[Drug]$

#### We can solve $\frac{d[Drug](t)}{dt}=-k[Drug](t)$ numerically using solve_ivp() from SciPy library.

The great thing about SciPy *solve_ivp()* is that you don't have to implement complex numerical algorithms yourself as we did above because *solve_ivp()* already has many algorithms implemented and you can just use them without fully understanding how they work but hopefully you have some idea from discussion above.

In [ ]:
solve_ivp?

In [ ]:
Drug_0 = [1]
tspan = [0, 5]

# Use solve_ivp to get numerical estimate of solved differential equation
soln = # TODO: calculate numerical solution wiht solve_ivp

plt.plot(soln.t, soln.y[0], label="solve_ivp() solution")
plt.legend();

Now, let's write code to solve $\frac{d[Drug]}{dt}=-k[Drug]$ without using solve_ivp() function with the goal of better understanding how solve_ivp() and similar methods actually works.

# Solving the differentially equation analytically

Differential equation $\frac{d[Drug]}{dt}=-k[Drug]$ is simple enough we can solve equation analytically using dsolve() function from SymPy library

In [ ]:
# Define symbols
Drug_ = sym.symbols("[Drug]", cls=sym.Function)
t, k, Drug_0 = sym.symbols("t, k, [Drug]_0")

# Define differential equation
diffeq = Eq(sym.diff(Drug_(t), t), - k * Drug_(t))
diffeq

In [ ]:
# Solve differential equation for Drug(t) for initial condition [Drug]_0
sym.dsolve(diffeq, Drug_(t), ics={Drug_(0): Drug_0})

# Comparing Euler method with analytical solution and solve_ivp() solution

In [ ]:
k = 1

plt.plot(time, np.exp(-k*time), label='[Drug]=exp(-t)')
plt.plot(soln.t, soln.y[0], label="solve_ivp() solution")
plt.plot(time, Drug, label='Euler')
plt.legend();

Curves for solve_ivp() and analytical solution are indistinguishable above and Euler at very low $\Delta t$ also merges with them

# Error analysis of solve_ivp

We are pretty close to the analytical solution but how close are we?

In [ ]:
# Plot the error of solve_ipv()
Drug_0 = [1]
tspan = [0, 5]

soln = solve_ivp(clearance_rate, tspan, Drug_0, t_eval=np.linspace(0, 5, 1000))

plt.plot(soln.t, soln.y[0]-np.exp(-soln.t), label="solve_ivp() error")
plt.legend();

### Source of errors in Euler and other numerical methods

**Runge-Kutta methods were developed in early 1900s!**   

**Many more methods have been described (and are continuing to be described) to numerically solve differential equations including higher order Runge-Kutta methods as well as completely different methods.**  
  
**You can take whole courses describing various numerical methods which are beyond the scope of this one lecture but the principle is similar to Euler, Taylor and Runge-Kutta methods described above.**